In [1]:
import sunpy
import numpy as np
from math import *
import astropy.units as u
from astropy.io import fits
import matplotlib.pyplot as plt
from sunpy.coordinates import frames
import matplotlib.gridspec as gridspec
from scipy.interpolate import interp1d,RegularGridInterpolator
from astropy.coordinates import SkyCoord
from scipy.io import readsav
from matplotlib.patches import Rectangle
from scipy.ndimage import zoom
from astropy.wcs import WCS
from astropy.wcs.utils import pixel_to_skycoord
import cv2
import glob
import os
from astropy.coordinates import SkyCoord
from sunpy.physics.differential_rotation import differential_rotate
from sunpy.coordinates import propagate_with_solar_surface
import pandas as pd

In [5]:
print(np.ceil(np.random.rand(5)*15))

[14. 13. 12.  4.  6.]


In [3]:
#裁剪目标区域并进行旋转对齐
root_dir=r'C:\Learning\PHD2nd\sunspotscar\data\M'
file_time=os.listdir(root_dir)
df=pd.read_excel('../dataset/M级耀斑.xlsx')
#df.drop(df.index[8:16], inplace=True)
x_range=df['X, arcsec'].values
y_range=df['Y, arcsec'].values
passbands=['131','211','304','hmi.Ic_45s']
width=800
for i in range(59,100):
    for j in passbands:
        file_name=glob.glob(os.path.join(root_dir,file_time[i],j,'*.fits'))
        os.makedirs(os.path.join(root_dir,file_time[i],j+'sub_map'), exist_ok=True)
        sub_file_path=os.path.join(root_dir,file_time[i],j+'sub_map')
        for k in range(len(file_name)):
            if k==0:
                map0=sunpy.map.Map(file_name[k])
                bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map0.coordinate_frame)
                tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map0.coordinate_frame)
                sub_map0=map0.submap(bl1,top_right=tr1)
                sub_map0.meta.pop('BLANK', None)
                sub_map0.save(os.path.join(sub_file_path,file_name[k][-38:]),overwrite=True)
            else:
                map1=sunpy.map.Map(file_name[k])
                with propagate_with_solar_surface():
                    rot_map1=map1.reproject_to(sub_map0.wcs,preserve_date_obs=True)
                for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                            'bunit', 'exptime']:
                    if key in map1.meta:
                        rot_map1.meta[key] = map1.meta[key]

                rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
                rot_map1.meta.pop('BLANK', None)
                rot_map1.save(os.path.join(sub_file_path, file_name[k][-38:]), overwrite=True)

#裁剪Br
for i in range(59,100):
    sub_file_path=os.path.join(root_dir,file_time[i],'hmi.B_720s')
    try:
        map0=sunpy.map.Map(sub_file_path+'\\'+'Br.fits')
        bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map0.coordinate_frame)
        tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map0.coordinate_frame)
        sub_map=map0.submap(bl1,top_right=tr1)
        sub_map.meta.pop('BLANK', None)
        sub_map.save(os.path.join(root_dir,file_time[i],'hmi.B_720s','Br_sub.fits'),overwrite=True)
    except:
        print(file_time[i]+'hmi.B_720s'+'Br.fits'+'不存在')

KeyboardInterrupt: 

In [4]:
from sub_rot_map_fast import run_sub_rot_map

report = run_sub_rot_map(
    root_dir=r"C:\Learning\PHD2nd\sunspotscar\data\M",
    excel_path="../dataset/M级耀斑.xlsx",
    n_events=100,
    max_workers=1,
    overwrite=False,
    cleanup_originals=True,
)

[000] saved_total=0 deleted_total=0
  20120701_191100_UTC/131: originals already cleaned
  20120701_191100_UTC/211: originals already cleaned
  20120701_191100_UTC/304: originals already cleaned
  20120701_191100_UTC/hmi.Ic_45s: originals already cleaned
  20120701_191100_UTC/hmi.B_720s: Br_sub exists
[001] saved_total=0 deleted_total=0
  20120702_002600_UTC/131: originals already cleaned
  20120702_002600_UTC/211: originals already cleaned
  20120702_002600_UTC/304: originals already cleaned
  20120702_002600_UTC/hmi.Ic_45s: originals already cleaned
  20120702_002600_UTC/hmi.B_720s: Br_sub exists
[002] saved_total=0 deleted_total=0
  20120702_104300_UTC/131: originals already cleaned
  20120702_104300_UTC/211: originals already cleaned
  20120702_104300_UTC/304: originals already cleaned
  20120702_104300_UTC/hmi.Ic_45s: originals already cleaned
  20120702_104300_UTC/hmi.B_720s: Br_sub exists
[003] saved_total=0 deleted_total=0
  20120702_195900_UTC/131: originals already cleaned
  

[018] saved_total=0 deleted_total=0
  20130422_102200_UTC/131: originals already cleaned
  20130422_102200_UTC/211: originals already cleaned
  20130422_102200_UTC/304: originals already cleaned
  20130422_102200_UTC/hmi.Ic_45s: ERROR Failed to read C:\Learning\PHD2nd\sunspotscar\data\M\20130422_102200_UTC\hmi.Ic_45s\hmi.ic_45s.20130422_095445_TAI.2.continuum.fits
buffer is too small for requested array
 If you want to bypass these errors, pass `allow_errors=True`.
  20130422_102200_UTC/hmi.B_720s: Br_sub exists
[019] saved_total=0 deleted_total=0
  20130502_045800_UTC/131: originals already cleaned
  20130502_045800_UTC/211: originals already cleaned
  20130502_045800_UTC/304: originals already cleaned
  20130502_045800_UTC/hmi.Ic_45s: originals already cleaned
  20130502_045800_UTC/hmi.B_720s: Br_sub exists
[020] saved_total=0 deleted_total=0
  20131013_001200_UTC/131: originals already cleaned
  20131013_001200_UTC/211: originals already cleaned
  20131013_001200_UTC/304: originals 

[050] saved_total=0 deleted_total=0
  20140202_180500_UTC/131: originals already cleaned
  20140202_180500_UTC/211: originals already cleaned
  20140202_180500_UTC/304: originals already cleaned
  20140202_180500_UTC/hmi.Ic_45s: ERROR Failed to read C:\Learning\PHD2nd\sunspotscar\data\M\20140202_180500_UTC\hmi.Ic_45s\hmi.ic_45s.20140202_173730_TAI.2.continuum.fits
buffer is too small for requested array
 If you want to bypass these errors, pass `allow_errors=True`.
  20140202_180500_UTC/hmi.B_720s: Br_sub exists
[051] saved_total=0 deleted_total=0
  20140202_212400_UTC/131: originals already cleaned
  20140202_212400_UTC/211: originals already cleaned
  20140202_212400_UTC/304: originals already cleaned
  20140202_212400_UTC/hmi.Ic_45s: originals already cleaned
  20140202_212400_UTC/hmi.B_720s: Br_sub exists
[052] saved_total=0 deleted_total=0
  20140204_011600_UTC/131: originals already cleaned
  20140204_011600_UTC/211: originals already cleaned
  20140204_011600_UTC/304: originals 

[057] saved_total=0 deleted_total=0
  20140211_163400_UTC/131: originals already cleaned
  20140211_163400_UTC/211: originals already cleaned
  20140211_163400_UTC/304: originals already cleaned
  20140211_163400_UTC/hmi.Ic_45s: originals already cleaned
  20140211_163400_UTC/hmi.B_720s: Br_sub exists


[058] saved_total=0 deleted_total=0
  20140212_035200_UTC/131: originals already cleaned
  20140212_035200_UTC/211: originals already cleaned
  20140212_035200_UTC/304: originals already cleaned
  20140212_035200_UTC/hmi.Ic_45s: ERROR No usable reference FITS for 20140212_035200_UTC/hmi.Ic_45s
  20140212_035200_UTC/hmi.B_720s: Br_sub exists


[059] saved_total=66 deleted_total=66
  20140212_065400_UTC/131: saved 66/66
  20140212_065400_UTC/131: deleted originals 66
  20140212_065400_UTC/211: ERROR No usable reference FITS for 20140212_065400_UTC/211
  20140212_065400_UTC/304: ERROR No usable reference FITS for 20140212_065400_UTC/304
  20140212_065400_UTC/hmi.Ic_45s: ERROR No usable reference FITS for 20140212_065400_UTC/hmi.Ic_45s
  20140212_065400_UTC/hmi.B_720s: Br_sub exists
[060] saved_total=0 deleted_total=0
  20140212_154100_UTC/131: ERROR No usable reference FITS for 20140212_154100_UTC/131
  20140212_154100_UTC/211: ERROR No usable reference FITS for 20140212_154100_UTC/211
  20140212_154100_UTC/304: ERROR No usable reference FITS for 20140212_154100_UTC/304
  20140212_154100_UTC/hmi.Ic_45s: no fits
  20140212_154100_UTC/hmi.B_720s: Br.fits not found


[061] saved_total=0 deleted_total=66
  20140213_013200_UTC/131: cutouts already exist
  20140213_013200_UTC/131: deleted originals 66
  20140213_013200_UTC/211: ERROR No usable reference FITS for 20140213_013200_UTC/211
  20140213_013200_UTC/304: ERROR No usable reference FITS for 20140213_013200_UTC/304
  20140213_013200_UTC/hmi.Ic_45s: ERROR No usable reference FITS for 20140213_013200_UTC/hmi.Ic_45s
  20140213_013200_UTC/hmi.B_720s: Br_sub exists
[062] saved_total=0 deleted_total=0
  20140213_024100_UTC/131: ERROR No usable reference FITS for 20140213_024100_UTC/131
  20140213_024100_UTC/211: ERROR No usable reference FITS for 20140213_024100_UTC/211
  20140213_024100_UTC/304: ERROR No usable reference FITS for 20140213_024100_UTC/304
  20140213_024100_UTC/hmi.Ic_45s: ERROR No usable reference FITS for 20140213_024100_UTC/hmi.Ic_45s
  20140213_024100_UTC/hmi.B_720s: Br_sub exists
[063] saved_total=0 deleted_total=0
  20140213_054900_UTC/131: ERROR No usable reference FITS for 201402

[067] saved_total=122 deleted_total=122
  20140214_122900_UTC/131: ERROR No usable reference FITS for 20140214_122900_UTC/131
  20140214_122900_UTC/211: ERROR No usable reference FITS for 20140214_122900_UTC/211
  20140214_122900_UTC/304: saved 67/67
  20140214_122900_UTC/304: deleted originals 67
  20140214_122900_UTC/hmi.Ic_45s: saved 55/55
  20140214_122900_UTC/hmi.Ic_45s: deleted originals 55
  20140214_122900_UTC/hmi.B_720s: Br_sub exists
[068] saved_total=255 deleted_total=255
  20140214_132100_UTC/131: saved 67/67
  20140214_132100_UTC/131: deleted originals 67
  20140214_132100_UTC/211: saved 67/67
  20140214_132100_UTC/211: deleted originals 67
  20140214_132100_UTC/304: saved 67/67
  20140214_132100_UTC/304: deleted originals 67
  20140214_132100_UTC/hmi.Ic_45s: saved 54/54
  20140214_132100_UTC/hmi.Ic_45s: deleted originals 54
  20140214_132100_UTC/hmi.B_720s: Br_sub exists
[069] saved_total=256 deleted_total=256
  20140216_092000_UTC/131: saved 67/67
  20140216_092000_UTC/1

[073] saved_total=188 deleted_total=188
  20140416_195400_UTC/131: saved 67/67
  20140416_195400_UTC/131: deleted originals 67
  20140416_195400_UTC/211: saved 67/67
  20140416_195400_UTC/211: deleted originals 67
  20140416_195400_UTC/304: ERROR No usable reference FITS for 20140416_195400_UTC/304
  20140416_195400_UTC/hmi.Ic_45s: saved 54/54
  20140416_195400_UTC/hmi.Ic_45s: deleted originals 54
  20140416_195400_UTC/hmi.B_720s: Br_sub exists
[074] saved_total=255 deleted_total=255
  20140603_035800_UTC/131: saved 67/67
  20140603_035800_UTC/131: deleted originals 67
  20140603_035800_UTC/211: saved 67/67
  20140603_035800_UTC/211: deleted originals 67
  20140603_035800_UTC/304: saved 67/67
  20140603_035800_UTC/304: deleted originals 67
  20140603_035800_UTC/hmi.Ic_45s: saved 54/54
  20140603_035800_UTC/hmi.Ic_45s: deleted originals 54
  20140603_035800_UTC/hmi.B_720s: Br_sub exists
[075] saved_total=255 deleted_total=255
  20140606_192600_UTC/131: saved 67/67
  20140606_192600_UTC/

In [5]:
from pathlib import Path
import pandas as pd
from sub_rot_map_fast import process_event

root_dir = Path(r"C:\Learning\PHD2nd\sunspotscar\data\M")
excel_path = "../dataset/M级耀斑.xlsx"

event_dirs = sorted(path for path in root_dir.iterdir() if path.is_dir())
df = pd.read_excel(excel_path)

x_values = df["X, arcsec"].to_numpy()
y_values = df["Y, arcsec"].to_numpy()

passbands = ("131", "211", "304", "hmi.Ic_45s")

for index in range(100, len(event_dirs)):
    event_dir = event_dirs[index]

    result = process_event(
        index,
        event_dir,
        float(x_values[index]),
        float(y_values[index]),
        passbands=passbands,
        width_arcsec=800,
        source_margin_arcsec=180,
        overwrite=False,
        crop_br=True,
        cleanup_originals=True,
    )

    print(result[0], "saved_total=", result[1], "deleted_total=", result[2])
    for msg in result[3]:
        print(" ", msg)

100 saved_total= 257 deleted_total= 256
  20150130_052900_UTC/131: saved 67/67
  20150130_052900_UTC/131: deleted originals 67
  20150130_052900_UTC/211: saved 67/67
  20150130_052900_UTC/211: deleted originals 67
  20150130_052900_UTC/304: saved 67/67
  20150130_052900_UTC/304: deleted originals 67
  20150130_052900_UTC/hmi.Ic_45s: saved 55/55
  20150130_052900_UTC/hmi.Ic_45s: deleted originals 55
  20150130_052900_UTC/hmi.B_720s: saved Br_sub.fits
101 saved_total= 257 deleted_total= 256
  20150204_020800_UTC/131: saved 67/67
  20150204_020800_UTC/131: deleted originals 67
  20150204_020800_UTC/211: saved 67/67
  20150204_020800_UTC/211: deleted originals 67
  20150204_020800_UTC/304: saved 67/67
  20150204_020800_UTC/304: deleted originals 67
  20150204_020800_UTC/hmi.Ic_45s: saved 55/55
  20150204_020800_UTC/hmi.Ic_45s: deleted originals 55
  20150204_020800_UTC/hmi.B_720s: saved Br_sub.fits
102 saved_total= 256 deleted_total= 255
  20150310_234600_UTC/131: saved 67/67
  20150310_23

KeyboardInterrupt: 

下面这些是看看对齐的效果

In [ ]:
ssmap0=sunpy.map.Map(r'C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\131sub_map\2012-07-12T150710Z.131.image_lev1.fits')
im0=ssmap0.plot(cmap='sdoaia131')
plt.colorbar(im0)

In [ ]:
ssmap1=sunpy.map.Map(r'C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\131sub_map\2012-07-12T150746Z.131.image_lev1.fits')
im1=ssmap1.plot(cmap='sdoaia131')
plt.colorbar(im1)

In [ ]:
map1=sunpy.map.Map(r'C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\131\aia.lev1_euv_12s.2012-07-12T150710Z.131.image_lev1.fits')
map2=sunpy.map.Map(r'C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\131\aia.lev1_euv_12s.2012-07-12T154646Z.131.image_lev1.fits')

In [ ]:
bl1=SkyCoord(Tx=(63.3-100)*u.arcsec,Ty=-436.8*u.arcsec,frame=map1.coordinate_frame)
tr1=SkyCoord(Tx=163.3*u.arcsec,Ty=-266*u.arcsec,frame=map1.coordinate_frame)
smap1=map1.submap(bl1,top_right=tr1)
smap1.plot(cmap='sdoaia131')

In [ ]:
with propagate_with_solar_surface():
    rot_map22=map2.reproject_to(smap1.wcs,preserve_date_obs=True)
rot_map22.plot(cmap='sdoaia131')

In [ ]:
#--------------------------------临时处理一些之前没处理好的文件-------------------------

In [13]:
#这次事件的Br_sub.fits有问题，临时裁剪一下
root_dir_br=r'C:\Learning\PHD2nd\sunspotscar\data\M\20140212_154100_UTC\hmi.Ic_45s'
map1=sunpy.map.Map(os.path.join(root_dir_br,'hmi.Ic_45s.20140212_160300_TAI.2.continuum.fits'))
width=800
i=60
bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map1.coordinate_frame)
tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map1.coordinate_frame)
smap=map1.submap(bl1,top_right=tr1)
smap.meta.pop('BLANK', None)
smap.save(os.path.join(root_dir_br,'Ic_smap.fits'), overwrite=True)

In [16]:
#裁剪目标区域并进行旋转对齐
root_dir=r'C:\Learning\PHD2nd\sunspotscar\data\M'
file_time=os.listdir(root_dir)
df=pd.read_excel('../dataset/M级耀斑.xlsx')
#df.drop(df.index[8:16], inplace=True)
x_range=df['X, arcsec'].values
y_range=df['Y, arcsec'].values
#passbands=['131']
passbands=['131','211','304','hmi.Ic_45s']
width=800
for i in range(66,68):
    # if i==61:
    #     passbands=['211','304','hmi.Ic_45s']
    if i==67:
        passbands=['131','211']
    for j in passbands:
        file_name=glob.glob(os.path.join(root_dir,file_time[i],j,'*.fits'))
        os.makedirs(os.path.join(root_dir,file_time[i],j+'sub_map'), exist_ok=True)
        sub_file_path=os.path.join(root_dir,file_time[i],j+'sub_map')
        for k in range(len(file_name)):
            if k==0:
                map0=sunpy.map.Map(file_name[k])
                bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map0.coordinate_frame)
                tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map0.coordinate_frame)
                sub_map0=map0.submap(bl1,top_right=tr1)
                sub_map0.meta.pop('BLANK', None)
                sub_map0.save(os.path.join(sub_file_path,file_name[k][-38:]),overwrite=True)
            else:
                map1=sunpy.map.Map(file_name[k])
                with propagate_with_solar_surface():
                    rot_map1=map1.reproject_to(sub_map0.wcs,preserve_date_obs=True)
                for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                            'bunit', 'exptime']:
                    if key in map1.meta:
                        rot_map1.meta[key] = map1.meta[key]

                rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
                rot_map1.meta.pop('BLANK', None)
                rot_map1.save(os.path.join(sub_file_path, file_name[k][-38:]), overwrite=True)

#裁剪Br
# for i in range(75,76):
#     sub_file_path=os.path.join(root_dir,file_time[i],'hmi.B_720s')
#     try:
#         map0=sunpy.map.Map(sub_file_path+'\\'+'Br.fits')
#         bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map0.coordinate_frame)
#         tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map0.coordinate_frame)
#         sub_map=map0.submap(bl1,top_right=tr1)
#         sub_map.meta.pop('BLANK', None)
#         sub_map.save(os.path.join(root_dir,file_time[i],'hmi.B_720s','Br_sub.fits'),overwrite=True)
#     except:
#         print(file_time[i]+'hmi.B_720s'+'Br.fits'+'不存在')

In [15]:
a=13992+30595+31604+13856+19660+42715+22189+20018+18248+33873+36848+34288+14747+19855+21063
print(a)
print(a/16)

373551
23346.9375


In [ ]:
from pathlib import Path
import pandas as pd
from sub_rot_map_fast import process_event

root_dir = Path(r"C:\Learning\PHD2nd\sunspotscar\data\M")
excel_path = "../dataset/M级耀斑.xlsx"

event_dirs = sorted(path for path in root_dir.iterdir() if path.is_dir())
df = pd.read_excel(excel_path)

x_values = df["X, arcsec"].to_numpy()
y_values = df["Y, arcsec"].to_numpy()

passbands = ("131", "211", "304", "hmi.Ic_45s")

for index in range(100, len(event_dirs)):
    event_dir = event_dirs[index]

    result = process_event(
        index,
        event_dir,
        float(x_values[index]),
        float(y_values[index]),
        passbands=passbands,
        width_arcsec=800,
        source_margin_arcsec=180,
        overwrite=False,
        crop_br=True,
        cleanup_originals=True,
    )

    print(result[0], "saved_total=", result[1], "deleted_total=", result[2])
    for msg in result[3]:
        print(" ", msg)

In [ ]:
#------------------------------------------------------之前的Br都没有旋转对齐就裁剪了，我现在针对这个再改一下-----------------------------------------------

In [ ]:
#裁剪Br incli并对目标区域进行旋转对齐+裁剪,采用131的第一张图对其进行对齐
root_dir=r'C:\Learning\PHD2nd\sunspotscar\data\M'
file_time=os.listdir(root_dir)

df=pd.read_excel('../dataset/M级耀斑.xlsx')
#df.drop(df.index[8:16], inplace=True)
x_range=df['X, arcsec'].values
y_range=df['Y, arcsec'].values
cnt=0
width=800
for i in range(len(file_time)):
    cnt=cnt+1
    print(cnt)
    #画出每个事件最早的131并进行裁剪
    try:
        ref_dir=os.path.join(root_dir,file_time[i],'131sub_map')
        ref_list=glob.glob(os.path.join(ref_dir,'*.fits'))
        map0=sunpy.map.Map(ref_list[0])
        # sub_map0=map0
        bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map0.coordinate_frame)
        tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map0.coordinate_frame)
        sub_map0=map0.submap(bl1,top_right=tr1)

        map1=sunpy.map.Map(os.path.join(root_dir,file_time[i],'hmi.B_720s','Br.fits'))
        map2=sunpy.map.Map(os.path.join(root_dir,file_time[i],'hmi.B_720s','inclination_calc.fits'))
        with propagate_with_solar_surface():
            rot_sub_map1=map1.reproject_to(sub_map0.wcs,preserve_date_obs=True)
            rot_sub_map2=map2.reproject_to(sub_map0.wcs,preserve_date_obs=True)
        rot_sub_map1.meta.pop('BLANK', None)
        rot_sub_map2.meta.pop('BLANK', None)

        rot_sub_map1.save(os.path.join(root_dir,file_time[i],'hmi.B_720s','Br_sub.fits'),overwrite=True)
        rot_sub_map2.save(os.path.join(root_dir,file_time[i],'hmi.B_720s','inclination_calc_sub.fits'),overwrite=True)
    except Exception as e:
        print(e)


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
Failed to read C:\Learning\PHD2nd\sunspotscar\data\M\20140213_013200_UTC\131sub_map\aia.lev1_euv_12s.2014-02-13T010210Z.131.image_lev1.fits
buffer is too small for requested array
 If you want to bypass these errors, pass `allow_errors=True`.
63
Failed to read C:\Learning\PHD2nd\sunspotscar\data\M\20140213_024100_UTC\hmi.B_720s\inclination_calc.fits
buffer is too small for requested array
 If you want to bypass these errors, pass `allow_errors=True`.
64


65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
Did not find any files at C:\Learning\PHD2nd\sunspotscar\data\M\20150311_071000_UTC\hmi.B_720s\Br.fits
105
Did not find any files at C:\Learning\PHD2nd\sunspotscar\data\M\20150311_075100_UTC\hmi.B_720s\Br.fits
106
Did not find any files at C:\Learning\PHD2nd\sunspotscar\data\M\20150311_183700_UTC\hmi.B_720s\Br.fits
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
list index out of range


In [ ]:
#这一小段是有两个事件没裁剪成功，临时修改的
x_range=[197.5,213.7]
y_range=[-80.1,-80.6]
cnt=0
width=800
i=1
try:
    ref_dir=r'C:\Learning\PHD2nd\sunspotscar\data\M\20140213_024100_UTC\131sub_map'
    ref_list=glob.glob(os.path.join(ref_dir,'*.fits'))
    map0=sunpy.map.Map(ref_list[1])
    bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map0.coordinate_frame)
    tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map0.coordinate_frame)
    sub_map0=map0.submap(bl1,top_right=tr1)


    map2=sunpy.map.Map(r'C:\Learning\PHD2nd\sunspotscar\data\M\20140213_024100_UTC\hmi.B_720s\inclination_calc.fits')
    with propagate_with_solar_surface():

        rot_sub_map2=map2.reproject_to(sub_map0.wcs,preserve_date_obs=True)

    rot_sub_map2.meta.pop('BLANK', None)

    rot_sub_map2.save(r'C:\Learning\PHD2nd\sunspotscar\data\M\20140213_024100_UTC\hmi.B_720s\inclination_calc_sub.fits',overwrite=True)
except Exception as e:
    print(e)


In [3]:
root_dir=r'C:\Learning\PHD2nd\sunspotscar\data\M'
file_time=os.listdir(root_dir)
for i in range(len(file_time)):
    bp=os.path.join(root_dir,file_time[i],'hmi.B_720s','Bp.fits')
    bt=os.path.join(root_dir,file_time[i],'hmi.B_720s','Bt.fits')
    if os.path.exists(bp):
        os.remove(bp)
    if os.path.exists(bt):
        os.remove(bt)

In [4]:
root_dir=r'D:\C\20120701_154100_UTC'
file_time=os.listdir(root_dir)
aia131=os.path.join(root_dir,'131')
import shutil 
shutil.rmtree(aia131)
# aia131=os.path.
# for i in range(len(file_time)):
#     bp=os.path.join(root_dir,file_time[i],'hmi.B_720s','Bp.fits')
#     bt=os.path.join(root_dir,file_time[i],'hmi.B_720s','Bt.fits')
#     if os.path.exists(bp):
#         os.remove(bp)
#     if os.path.exists(bt):
#         os.remove(bt)